<style>
:root {
  --navy:#071022; --navy2:#0d1b35; --panel:#111f3d; --line:#263b66;
  --blue:#3b82f6; --cyan:#22d3ee; --green:#22c55e; --red:#ef4444; --yellow:#facc15;
  --text:#e5e7eb; --muted:#94a3b8;
}
.guide-hero {
  background: linear-gradient(135deg,#061026 0%,#0f1f3f 55%,#172554 100%);
  border:1px solid #263b66; border-radius:18px; padding:28px 32px; color:#e5e7eb;
  box-shadow: 0 18px 45px rgba(15,23,42,.28);
}
.guide-hero h1 {font-size:32px; margin:0 0 8px 0; line-height:1.15;}
.guide-hero p {font-size:16px; color:#b6c4dc; margin:0;}
.badges {margin-top:18px; display:flex; gap:8px; flex-wrap:wrap;}
.badge {display:inline-block; padding:6px 10px; border-radius:999px; background:#172554; border:1px solid #2c477a; color:#dbeafe; font-size:12px; font-weight:700;}
.card {background:#0f172a; border:1px solid #263b66; border-radius:14px; padding:16px 18px; margin:14px 0; color:#e5e7eb;}
.card-blue {border-left:5px solid #3b82f6;}
.card-green {border-left:5px solid #22c55e;}
.card-red {border-left:5px solid #ef4444;}
.card-yellow {border-left:5px solid #facc15;}
.card-purple {border-left:5px solid #a855f7;}
.small {color:#94a3b8; font-size:14px;}
.kbd {font-family:ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; background:#020617; border:1px solid #334155; color:#e2e8f0; border-radius:6px; padding:2px 6px; font-size:12px;}
.file {font-family:ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; color:#93c5fd; font-weight:700;}
.flow {font-family:ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; background:#020617; border:1px dashed #334155; color:#cbd5e1; border-radius:12px; padding:16px; line-height:1.8; white-space:pre-wrap;}
table {border-collapse:collapse !important; width:100%;}
th {background:#0f1f3f !important; color:#e5e7eb !important; border:1px solid #263b66 !important; padding:10px !important;}
td {border:1px solid #263b66 !important; padding:10px !important; vertical-align:top !important;}
.callout {background:#061425; border-left:5px solid #22d3ee; padding:14px 16px; border-radius:10px; margin:12px 0;}
.warn {background:#211609; border-left:5px solid #f59e0b; padding:14px 16px; border-radius:10px; margin:12px 0;}
.danger {background:#210c12; border-left:5px solid #ef4444; padding:14px 16px; border-radius:10px; margin:12px 0;}
.ok {background:#052114; border-left:5px solid #22c55e; padding:14px 16px; border-radius:10px; margin:12px 0;}
hr {border:0; border-top:1px solid #263b66; margin:28px 0;}
</style>

<div class="guide-hero">
  <h1>AIC2026 Retrieval Frontend — Source Code Guide</h1>
  <p>Cuốn hướng dẫn đọc source dành cho người mới nhận project: biết file nào làm gì, sửa ở đâu, luồng hoạt động ra sao, và cách debug khi lỗi.</p>
  <div class="badges">
    <span class="badge">React + TypeScript</span>
    <span class="badge">Vite</span>
    <span class="badge">Zustand Store</span>
    <span class="badge">Tailwind UI</span>
    <span class="badge">AIC2026 CSV</span>
    <span class="badge">Frontend Prototype</span>
  </div>
</div>

<div class="warn">
<b>Cách dùng notebook này:</b> đặt file <span class="file">.ipynb</span> này ở <b>root project</b>, cùng cấp với <span class="file">package.json</span> và thư mục <span class="file">src/</span>. Khi đó các link như <a href="src/App.tsx">src/App.tsx</a> có thể bấm để mở file trong VS Code/Jupyter.
</div>

# 0. Mục tiêu của frontend này

Frontend này là giao diện cho hệ thống truy xuất multimedia trong AIC2026. Người dùng nhập mô tả, câu hỏi hoặc chuỗi sự kiện; hệ thống hiển thị các frame/video liên quan; sau đó người dùng chọn kết quả tốt nhất vào cột Answer Builder để export CSV nộp lên hệ thống.

<div class="flow">
Người dùng nhập query
        ↓
SearchPanel nhận input + mode + topK + collection + model
        ↓
api.ts trả kết quả mock hoặc gọi backend thật sau này
        ↓
store.ts lưu results
        ↓
ResultGrid hiển thị danh sách ResultCard
        ↓
Người dùng preview hoặc đưa frame vào Answer Builder
        ↓
RecheckPanel xuất CSV theo KIS / QA / TRAKE
</div>

<div class="ok">
<b>Điểm quan trọng:</b> frontend hiện có thể chạy prototype không cần AI/database. Sau này team backend chỉ cần thay logic trong <a href="src/api.ts"><span class="file">src/api.ts</span></a> để gọi API thật.
</div>

# 1. Bản đồ source code

| Khu vực | File / thư mục | Vai trò chính |
|---|---|---|
| Entry point | [src/main.tsx](src/main.tsx) | Gắn React app vào <span class="kbd">#root</span> trong <span class="file">index.html</span>. |
| Layout tổng | [src/App.tsx](src/App.tsx) | Chia màn hình thành header, panel trái, kết quả giữa, panel phải, modal. |
| Style toàn cục | [src/index.css](src/index.css) | Tailwind import, màu nền, scrollbar, font, utility CSS. |
| State trung tâm | [src/store.ts](src/store.ts) | Lưu query, mode, kết quả, recheck list, modal đang mở, số cột, toast. |
| API layer | [src/api.ts](src/api.ts) | Chỗ nối backend sau này. Hiện có thể dùng mock data. |
| Mock data | [src/data/](src/data/) | Dữ liệu giả để frontend chạy khi chưa có backend. |
| Media helper | [src/utils/media.ts](src/utils/media.ts) | Helper tạo link ảnh/video hoặc normalize đường dẫn media. |
| Search UI | [src/components/SearchPanel.tsx](src/components/SearchPanel.tsx) | Ô nhập query, mode, topK, collection, model, advanced filters, history. |
| Grid kết quả | [src/components/ResultGrid.tsx](src/components/ResultGrid.tsx) | Vùng giữa, render danh sách kết quả hoặc empty/loading state. |
| Card kết quả | [src/components/ResultCard.tsx](src/components/ResultCard.tsx) | Mỗi frame kết quả: ảnh, score, videoId, frameId, nút preview, nút add. |
| Recheck/CSV | [src/components/RecheckPanel.tsx](src/components/RecheckPanel.tsx) | Cột phải Answer Builder, sắp xếp, xóa, copy/export CSV. |
| Preview ảnh | [src/components/PreviewModal.tsx](src/components/PreviewModal.tsx) | Modal xem ảnh lớn và metadata chi tiết. |
| Preview video | [src/components/VideoModal.tsx](src/components/VideoModal.tsx) | Modal xem video tại timestamp nếu backend media sẵn sàng. |

<div class="callout">
<b>Quy tắc đọc source:</b> đọc theo thứ tự <span class="file">main.tsx → App.tsx → store.ts → api.ts → SearchPanel → ResultGrid → ResultCard → RecheckPanel → PreviewModal</span>. Đọc ngược sẽ dễ rối vì component phụ thuộc state từ store.
</div>

# 2. Luồng hoạt động chính khi bấm Search

<div class="flow">
1. User nhập query ở SearchPanel
2. SearchPanel lấy query, mode, topK, collection, model từ store
3. User bấm SEARCH
4. SearchPanel gọi hàm search trong api.ts hoặc mock search
5. Trong lúc chờ, store bật isLoading = true
6. API trả danh sách frame result
7. store lưu results
8. ResultGrid tự render lại vì đang subscribe results từ store
9. Người dùng thấy các ResultCard ở giữa màn hình
</div>

| Bước | Nơi cần xem | Ghi chú |
|---|---|---|
| Nhập query | [SearchPanel.tsx](src/components/SearchPanel.tsx) | Tìm các biến query, setQuery, handleSearch. |
| Gọi dữ liệu | [api.ts](src/api.ts) | Sau này backend nối ở đây. |
| Lưu kết quả | [store.ts](src/store.ts) | Tìm state results/searchResults và setter tương ứng. |
| Hiển thị kết quả | [ResultGrid.tsx](src/components/ResultGrid.tsx) | Nếu results rỗng thì hiện empty state. |
| Hiển thị từng frame | [ResultCard.tsx](src/components/ResultCard.tsx) | Render ảnh, score, frameId, nút thao tác. |

<div class="warn">
Nếu bấm Search không thấy gì: kiểm tra lần lượt <span class="file">SearchPanel.tsx</span> có gọi search không, <span class="file">api.ts</span> có trả result không, và <span class="file">store.ts</span> có set results không.
</div>

# 3. Header và logo nằm ở đâu?

Logo, tên web, nút đổi màu, nút chỉnh số cột thường nằm trong:

<div class="card card-blue">
<a href="src/App.tsx"><span class="file">src/App.tsx</span></a><br>
<span class="small">Tìm khu vực <span class="kbd">header</span>, tên app như “News Retrieval v2”, hoặc chữ/logo ở góc trái.</span>
</div>

## Vai trò của header

| Thành phần | Làm gì | Sửa ở đâu |
|---|---|---|
| Logo | Hiển thị biểu tượng ở góc trái | [App.tsx](src/App.tsx) |
| Tên hệ thống | Ví dụ “News Retrieval v2” hoặc “AIC2026 Retrieval” | [App.tsx](src/App.tsx) |
| Set color | Đổi accent color cho một số nút/nhãn | [App.tsx](src/App.tsx), state trong [store.ts](src/store.ts) |
| COLS | Chỉnh số cột kết quả ở giữa | [App.tsx](src/App.tsx), render ở [ResultGrid.tsx](src/components/ResultGrid.tsx) |

<div class="callout">
Nếu dùng ảnh logo trong <span class="file">public/img/logo.jpg</span>, đường dẫn trong React nên là <span class="kbd">/img/logo.jpg</span>, không phải <span class="kbd">/public/img/logo.jpg</span>.
</div>

# 4. `store.ts` — bộ nhớ trung tâm của app

<div class="card card-purple">
<a href="src/store.ts"><span class="file">src/store.ts</span></a> là file quan trọng nhất. Nó dùng Zustand để lưu trạng thái dùng chung giữa nhiều component.
</div>

## Store đang giữ những nhóm dữ liệu nào?

| Nhóm state | Ý nghĩa | Component thường dùng |
|---|---|---|
| Query/search config | query, mode, topK, language, collection, model | SearchPanel |
| Results | danh sách frame tìm được | ResultGrid, ResultCard |
| Recheck list | danh sách frame đã chọn để nộp CSV | RecheckPanel, PreviewModal, ResultCard |
| Modal state | frame đang preview, video đang mở | App, PreviewModal, VideoModal |
| UI state | columns, accentColor, loading, error, toast | App, ResultGrid, SearchPanel |
| History | lịch sử query đã tìm | SearchPanel |

## Cách hiểu đơn giản

<div class="flow">
Component không tự giữ mọi thứ riêng lẻ.
Nó đọc/ghi vào store.
Khi store đổi, các component liên quan tự render lại.
</div>

Ví dụ tư duy:

| Khi người dùng... | Store thay đổi gì? | UI nào đổi theo? |
|---|---|---|
| Chọn mode QA | mode = QA | SearchPanel đổi tab active, Recheck export QA |
| Bấm Search | isLoading, results, history | ResultGrid hiện loading rồi hiện cards |
| Bấm mũi tên add | recheckItems thêm 1 item | RecheckPanel hiện item mới |
| Bấm card preview | selectedFrame đổi | PreviewModal mở |
| Bấm Clear | recheckItems = [] | RecheckPanel trống |

<div class="danger">
Không nên tạo state trùng lặp ở nhiều nơi. Ví dụ nếu <span class="file">recheckItems</span> đã nằm trong store, đừng tạo thêm một recheck list riêng trong RecheckPanel, vì sẽ dễ lệch dữ liệu.
</div>

# 5. `SearchPanel.tsx` — panel trái

<div class="card card-blue">
<a href="src/components/SearchPanel.tsx"><span class="file">src/components/SearchPanel.tsx</span></a> là nơi người dùng nhập query, chọn mode và filter.
</div>

## Thành phần chính trong SearchPanel

| Khu vực UI | Vai trò | Từ khóa nên search trong file |
|---|---|---|
| Mode tabs | AUTO / KIS / QA / TRAKE | <span class="kbd">modes</span>, <span class="kbd">setMode</span> |
| Query box | Textarea nhập mô tả/câu hỏi | <span class="kbd">textarea</span>, <span class="kbd">setQuery</span> |
| Search button | Kích hoạt tìm kiếm | <span class="kbd">handleSearch</span> |
| Top K | Giới hạn số result | <span class="kbd">topK</span>, <span class="kbd">setTopK</span> |
| Language | AUTO / VI / EN | <span class="kbd">language</span> |
| Collection | Chọn index/database | <span class="kbd">collection</span> |
| Model | Chọn CLIP/SigLIP/SigLIP2 | <span class="kbd">model</span> |
| Advanced filters | score, videoId, object, OCR, ASR | <span class="kbd">FilterBlock</span> |
| Search history | Lịch sử query đã chạy | <span class="kbd">history</span> |

## SearchPanel hoạt động như thế nào?

<div class="flow">
User nhập query
    ↓
setQuery lưu query vào store
    ↓
User chọn mode/topK/model/collection
    ↓
Các setter trong store cập nhật config
    ↓
User bấm SEARCH
    ↓
handleSearch gọi api.ts hoặc mock data
    ↓
setResults lưu kết quả
    ↓
addHistory lưu lịch sử
</div>

<div class="callout">
Nếu muốn thêm filter mới, nên thêm UI ở SearchPanel trước, sau đó bổ sung field vào request gửi backend trong <a href="src/api.ts"><span class="file">api.ts</span></a>. Đừng xử lý AI/database trong SearchPanel.
</div>

# 6. `api.ts` — cầu nối frontend và backend

<div class="card card-green">
<a href="src/api.ts"><span class="file">src/api.ts</span></a> là nơi sau này thay mock search bằng API thật.
</div>

## Vai trò của api.ts

Frontend không nên gọi lung tung nhiều endpoint trong nhiều component. Thay vào đó:

<div class="flow">
SearchPanel / ResultCard / PreviewModal
        ↓
api.ts
        ↓
Backend FastAPI / Flask / Node / service khác
        ↓
Vector DB / YOLO / OCR / ASR / LLM
</div>

## API backend nên trả về gì?

Mỗi result nên có tối thiểu:

| Field | Ý nghĩa |
|---|---|
| videoId | tên video, ví dụ L21_V041 |
| frameId | số frame/keyframe |
| timestampMs | thời điểm trong video, nếu có |
| imageUrl hoặc image_path | ảnh thumbnail/frame |
| score | điểm similarity/ranking |
| caption | caption mô tả frame, nếu có |
| ocr | text trong ảnh, nếu có |
| asr | transcript âm thanh, nếu có |
| objects | danh sách object YOLO, nếu có |

<div class="warn">
Khi nối backend thật, ưu tiên chỉ sửa <span class="file">api.ts</span>. Nếu phải sửa quá nhiều component mới chạy được, nghĩa là backend trả format chưa khớp với frontend.
</div>

# 7. `ResultGrid.tsx` — vùng kết quả ở giữa

<div class="card card-blue">
<a href="src/components/ResultGrid.tsx"><span class="file">src/components/ResultGrid.tsx</span></a> điều khiển vùng giữa màn hình.
</div>

## Nó làm gì?

| Trạng thái | UI hiển thị |
|---|---|
| isLoading = true | loading spinner / Searching... |
| results rỗng | thông báo “No results...” |
| results có dữ liệu | grid các ResultCard |

## Vì sao chỉnh số cột được?

Số cột lấy từ state <span class="kbd">columns</span> trong store. Header trong <a href="src/App.tsx"><span class="file">App.tsx</span></a> đổi columns, còn ResultGrid dùng giá trị đó để render grid.

<div class="flow">
Header bấm COLS 4
    ↓
store.columns = 4
    ↓
ResultGrid đọc columns
    ↓
Grid đổi thành 4 cột
</div>

# 8. `ResultCard.tsx` — mỗi frame kết quả

<div class="card card-yellow">
<a href="src/components/ResultCard.tsx"><span class="file">src/components/ResultCard.tsx</span></a> là card đại diện cho một frame/video keyframe.
</div>

## Mỗi card thường hiển thị

| Thành phần | Ý nghĩa |
|---|---|
| Thumbnail | ảnh frame/keyframe |
| Rank | thứ hạng trong kết quả search |
| Score | similarity score từ CLIP/SigLIP/backend |
| videoId | tên video |
| frameId | frame index |
| caption/object tags | metadata hỗ trợ người thi quyết định |
| Nút phóng to | mở PreviewModal |
| Nút mũi tên | thêm vào Answer Builder/Recheck |

## Hai thao tác quan trọng

<div class="flow">
Bấm hình vuông / preview
    ↓
setSelectedFrame(frame)
    ↓
PreviewModal mở

Bấm mũi tên qua phải
    ↓
addToRecheck(frame)
    ↓
RecheckPanel có thêm candidate
</div>

<div class="callout">
Nếu muốn đổi giao diện card, sửa ở đây. Nếu muốn đổi dữ liệu card nhận từ backend, kiểm tra thêm <span class="file">api.ts</span> và type/state trong <span class="file">store.ts</span>.
</div>

# 9. `RecheckPanel.tsx` — Answer Builder và export CSV

<div class="card card-red">
<a href="src/components/RecheckPanel.tsx"><span class="file">src/components/RecheckPanel.tsx</span></a> là file quan trọng cho phần nộp bài.
</div>

## RecheckPanel làm gì?

| Chức năng | Ý nghĩa |
|---|---|
| Hiển thị recheckItems | danh sách frame đã chọn từ ResultCard/PreviewModal |
| Rank # | thứ tự dự đoán khi nộp |
| Move up/down | đổi thứ tự ưu tiên |
| Delete | xóa candidate sai |
| Clear | xóa toàn bộ danh sách |
| Copy | copy nội dung CSV vào clipboard |
| CSV | tải file .csv để nộp |
| QA answer input | nhập câu trả lời cho mode QA |

## CSV phải xuất theo 3 dạng

| Mode | Format mỗi dòng | Ghi chú |
|---|---|---|
| KIS | <span class="kbd">videoId,frameId</span> | Tìm một khoảnh khắc cụ thể. |
| QA | <span class="kbd">videoId,frameId,answer</span> | Answer tối đa 100 ký tự, cần escape CSV nếu có dấu phẩy/ngoặc kép/xuống dòng. |
| TRAKE | <span class="kbd">videoId,frame1,frame2,...</span> | Chuỗi sự kiện theo thứ tự thời gian. |

<div class="danger">
File CSV nộp không có header. Không xuất Excel .xlsx. Không thêm dòng tiêu đề như <span class="kbd">video_id,frame_id</span> nếu BTC yêu cầu không header.
</div>

## Luồng export CSV

<div class="flow">
recheckItems trong store
    ↓
RecheckPanel đọc mode hiện tại hoặc answerType của từng item
    ↓
Tạo text CSV đúng format
    ↓
Nút Copy: navigator.clipboard.writeText(csv)
Nút CSV: tạo Blob và download .csv
</div>

# 10. `PreviewModal.tsx` — xem frame lớn

<div class="card card-purple">
<a href="src/components/PreviewModal.tsx"><span class="file">src/components/PreviewModal.tsx</span></a> là modal hiện khi người dùng bấm preview/phóng to frame.
</div>

## PreviewModal thường có

| Khu vực | Vai trò |
|---|---|
| Header | hiển thị rank/videoId/frameId/timestamp và nút đóng |
| Ảnh lớn | xem rõ frame để quyết định có nộp không |
| Add button | thêm frame vào Answer Builder |
| Overview tab | metadata tổng quan: score, caption, OCR, ASR, objects |
| Các tab khác | có thể mở rộng: neighbors, Q&A, TRAKE sequence, OCR |

## Khi nào PreviewModal mở?

<div class="flow">
ResultCard gọi setSelectedFrame(frame)
    ↓
store.selectedFrame có dữ liệu
    ↓
App.tsx thấy selectedFrame != null
    ↓
render PreviewModal
</div>

## Khi nào đóng?

<div class="flow">
User bấm X hoặc close
    ↓
setSelectedFrame(null)
    ↓
PreviewModal biến mất
</div>

# 11. `VideoModal.tsx` — xem video tại timestamp

<div class="card card-green">
<a href="src/components/VideoModal.tsx"><span class="file">src/components/VideoModal.tsx</span></a> dùng khi backend có thể serve video gốc.
</div>

## Vai trò

VideoModal cho phép người dùng xem video quanh frame đang chọn. Điều này hữu ích trong KIS/TRAKE vì một frame đơn có thể chưa đủ xác nhận ngữ cảnh.

| Cần backend cung cấp | Ví dụ |
|---|---|
| endpoint video | <span class="kbd">/api/media/video/L21_V041</span> |
| timestamp | <span class="kbd">1878837ms</span> |
| videoId | <span class="kbd">L21_V041</span> |

<div class="warn">
Nếu chưa có backend serve video, VideoModal có thể chưa hoạt động đầy đủ. Đây không phải lỗi frontend chính, mà là thiếu media endpoint.
</div>

# 12. `index.css` — màu sắc và style toàn cục

<div class="card card-blue">
<a href="src/index.css"><span class="file">src/index.css</span></a> chứa style nền tảng cho toàn app.
</div>

## Trong file này thường có

| Khu vực | Công dụng |
|---|---|
| Tailwind import | kích hoạt class Tailwind |
| Theme màu | navy, blue, red, accent color |
| Body style | nền tối, font, overflow |
| Scrollbar | thanh cuộn nhỏ, hợp UI tối |
| Utility class | badge, score color, animation |

## Khi muốn đổi tone màu toàn app

Ưu tiên sửa ở <span class="file">index.css</span> nếu là màu toàn cục. Nếu chỉ đổi một nút/khu vực cụ thể thì sửa trong component đó.

<div class="callout">
Nếu giao diện tự nhiên bị trắng hoặc class không ăn, kiểm tra Tailwind/Vite setup trước, sau đó xem <span class="file">index.css</span> có được import trong <span class="file">main.tsx</span> không.
</div>

# 13. `utils/media.ts` — xử lý đường dẫn ảnh/video

<div class="card card-yellow">
<a href="src/utils/media.ts"><span class="file">src/utils/media.ts</span></a> thường dùng để chuẩn hóa đường dẫn media.
</div>

## Tại sao cần file này?

Backend có thể trả về nhiều kiểu đường dẫn:

| Backend trả | Frontend cần dùng |
|---|---|
| <span class="kbd">L21_V041/83991.jpg</span> | convert thành URL ảnh đầy đủ |
| <span class="kbd">/static/frames/...</span> | dùng trực tiếp nếu đã serve tĩnh |
| <span class="kbd">http://...</span> | giữ nguyên |
| <span class="kbd">image_path</span> | map sang endpoint media |

Nếu mỗi component tự xử lý đường dẫn ảnh, code sẽ rối. Helper media giúp gom logic này lại một chỗ.

<div class="warn">
Khi backend đổi cách lưu ảnh, ưu tiên sửa <span class="file">media.ts</span> hoặc <span class="file">api.ts</span>, không nên sửa từng ResultCard/PreviewModal.
</div>

# 14. Mock data nằm ở đâu và dùng để làm gì?

<div class="card card-green">
Thư mục <a href="src/data/"><span class="file">src/data/</span></a> chứa dữ liệu giả để frontend chạy khi backend chưa xong.
</div>

## Vai trò của mock data

| Lý do dùng mock | Lợi ích |
|---|---|
| Chưa có vector database | vẫn test được UI search/result/recheck |
| Chưa có CLIP/SigLIP | vẫn demo được flow người dùng |
| Chưa có media server | có thể dùng ảnh URL mẫu hoặc public asset |
| Chưa có API thật | frontend không bị chờ backend |

<div class="callout">
Khi backend xong, không cần xóa mock data ngay. Có thể giữ lại để test offline hoặc fallback khi API lỗi.
</div>

# 15. Các mode AIC2026 trong frontend

Frontend cần hỗ trợ 4 tab/mode chính:

| Mode | Tên đầy đủ | Người dùng làm gì? | CSV export |
|---|---|---|---|
| AUTO | Tự đoán mode hoặc tìm tổng quát | Nhập query bất kỳ | thường fallback KIS hoặc theo item type |
| KIS | Known Item Search | Tìm đúng một khoảnh khắc/frame | <span class="kbd">videoId,frameId</span> |
| QA | Question Answering | Hỏi câu hỏi, chọn frame và nhập answer | <span class="kbd">videoId,frameId,answer</span> |
| TRAKE | Temporal Retrieval and Alignment of Key Events | Tìm chuỗi sự kiện theo thứ tự | <span class="kbd">videoId,frame1,frame2,...</span> |

## Nên chỉnh mode ở đâu?

| Muốn chỉnh | File |
|---|---|
| Tên tab mode bên trái | [SearchPanel.tsx](src/components/SearchPanel.tsx) |
| Giá trị mode trong state | [store.ts](src/store.ts) |
| Cách add item theo mode | [store.ts](src/store.ts), [ResultCard.tsx](src/components/ResultCard.tsx), [PreviewModal.tsx](src/components/PreviewModal.tsx) |
| Cách export CSV theo mode | [RecheckPanel.tsx](src/components/RecheckPanel.tsx) |

# 16. Backend sau này cần nối ở đâu?

<div class="ok">
Frontend đã chia sẵn để backend chỉ cần tập trung vào API. File quan trọng nhất để nối backend là <a href="src/api.ts"><span class="file">src/api.ts</span></a>.
</div>

## Contract tối thiểu backend nên có

| Endpoint | Method | Mục đích | Frontend dùng ở đâu |
|---|---|---|---|
| <span class="kbd">/api/search/</span> | POST | tìm frame theo query | SearchPanel → api.ts |
| <span class="kbd">/api/search/continuous</span> | POST | tìm frame gần giống frame đã chọn | ResultCard/PreviewModal nếu có |
| <span class="kbd">/api/media/image/{path}</span> | GET | serve ảnh frame | ResultCard/PreviewModal |
| <span class="kbd">/api/media/video/{videoId}</span> | GET | serve video | VideoModal |

## Backend AI/database dự kiến

<div class="flow">
Raw videos
   ↓ extract frames
Keyframes + metadata
   ↓ CLIP/SigLIP embedding
Vector database: Qdrant / Supabase pgvector / FAISS
   ↓ search query embedding
TopK frame results
   ↓ optional rerank: OCR/ASR/YOLO/LLM
JSON response to frontend
</div>

<div class="warn">
Frontend không chạy YOLO/CLIP trực tiếp trong browser ở bản prototype. Những phần nặng nên để backend xử lý, frontend chỉ nhận result và hiển thị.
</div>

# 17. Cách debug theo triệu chứng

| Triệu chứng | File cần kiểm tra | Cách nghĩ |
|---|---|---|
| Trang trắng | [main.tsx](src/main.tsx), [App.tsx](src/App.tsx), console browser | App có render không, import có lỗi không? |
| CSS không ăn | [index.css](src/index.css), [main.tsx](src/main.tsx), vite config | index.css có import không? Tailwind có chạy không? |
| Bấm Search không ra kết quả | [SearchPanel.tsx](src/components/SearchPanel.tsx), [api.ts](src/api.ts), [store.ts](src/store.ts) | handleSearch có chạy? API có trả data? setResults có gọi? |
| Có data nhưng không hiện card | [ResultGrid.tsx](src/components/ResultGrid.tsx), [ResultCard.tsx](src/components/ResultCard.tsx) | results có đúng format không? key/id có tồn tại không? |
| Ảnh không hiện | [ResultCard.tsx](src/components/ResultCard.tsx), [media.ts](src/utils/media.ts), backend media | imageUrl/path sai hay server chưa serve ảnh? |
| Add không vào Answer Builder | [ResultCard.tsx](src/components/ResultCard.tsx), [store.ts](src/store.ts), [RecheckPanel.tsx](src/components/RecheckPanel.tsx) | addToRecheck có gọi? recheckItems có tăng? |
| CSV sai format | [RecheckPanel.tsx](src/components/RecheckPanel.tsx) | đang export mode nào? có header không? answer có escape không? |
| Preview không mở | [ResultCard.tsx](src/components/ResultCard.tsx), [store.ts](src/store.ts), [PreviewModal.tsx](src/components/PreviewModal.tsx), [App.tsx](src/App.tsx) | selectedFrame có được set không? App có render modal không? |
| Người khác không vào localhost | [vite.config.ts](vite.config.ts), firewall, IP máy host | server.host phải là 0.0.0.0, dùng IP LAN đúng. |

# 18. Muốn sửa tính năng thì sửa file nào?

| Tôi muốn... | Sửa file chính | Có thể cần sửa thêm |
|---|---|---|
| Đổi logo/tên app | [App.tsx](src/App.tsx) | [index.css](src/index.css) nếu đổi style |
| Đổi màu chủ đạo | [App.tsx](src/App.tsx), [index.css](src/index.css) | [store.ts](src/store.ts) nếu màu là state |
| Thêm mode mới | [SearchPanel.tsx](src/components/SearchPanel.tsx) | [store.ts](src/store.ts), [RecheckPanel.tsx](src/components/RecheckPanel.tsx) |
| Thêm filter mới | [SearchPanel.tsx](src/components/SearchPanel.tsx) | [api.ts](src/api.ts), backend search request |
| Đổi layout result card | [ResultCard.tsx](src/components/ResultCard.tsx) | [ResultGrid.tsx](src/components/ResultGrid.tsx) |
| Đổi số cột mặc định | [store.ts](src/store.ts) | [App.tsx](src/App.tsx) nếu đổi nút chọn cột |
| Thêm metadata vào preview | [PreviewModal.tsx](src/components/PreviewModal.tsx) | [api.ts](src/api.ts), backend response |
| Đổi format CSV | [RecheckPanel.tsx](src/components/RecheckPanel.tsx) | [store.ts](src/store.ts) nếu cần thêm field |
| Nối backend thật | [api.ts](src/api.ts) | [media.ts](src/utils/media.ts), vite proxy nếu cần |
| Thay mock data | [src/data/](src/data/) | [api.ts](src/api.ts) |

# 19. Quy tắc giữ code dễ hiểu cho người sau

<div class="ok">
<b>1. Component chỉ lo UI.</b><br>
SearchPanel, ResultCard, RecheckPanel nên chủ yếu render UI và gọi hàm. Không nhét logic AI/database vào component.
</div>

<div class="ok">
<b>2. Store giữ state dùng chung.</b><br>
Nếu dữ liệu cần nhiều component cùng dùng, đưa vào store. Nếu chỉ là trạng thái nhỏ trong một component, dùng local state.
</div>

<div class="ok">
<b>3. API gom vào api.ts.</b><br>
Sau này đổi FastAPI endpoint hoặc thêm token/auth thì chỉ sửa api.ts là chính.
</div>

<div class="ok">
<b>4. Backend response phải ổn định.</b><br>
Frontend sẽ dễ maintain nếu backend luôn trả result theo cùng format.
</div>

<div class="danger">
<b>Tránh:</b> copy thêm một store mới, tạo nhiều kiểu data result khác nhau, hoặc để mỗi component tự gọi backend một kiểu. Những thứ này làm app rất khó debug khi thi.
</div>

# 20. Checklist khi nhận source lần đầu

- [ ] Chạy <span class="kbd">npm install</span> thành công.
- [ ] Chạy <span class="kbd">npm run dev</span> thấy giao diện hiện lên.
- [ ] Nhập query và bấm Search có result mock hoặc API.
- [ ] Bấm card preview mở modal.
- [ ] Bấm mũi tên thêm được vào Answer Builder.
- [ ] Move up/down trong Answer Builder hoạt động.
- [ ] Copy CSV hoạt động.
- [ ] Export CSV tải file .csv đúng format.
- [ ] Đổi mode KIS/QA/TRAKE không làm crash app.
- [ ] Đổi số cột COLS hoạt động.
- [ ] Logo hiển thị đúng.
- [ ] Console browser không có lỗi đỏ nghiêm trọng.

<div class="callout">
Khi bàn giao cho backend, gửi kèm file này và nói rõ: “Frontend cần backend trả result đúng contract trong phần 16. Trước mắt chỉ cần sửa <span class="file">api.ts</span> để nối API thật.”
</div>

# 21. Tóm tắt cực ngắn để nhớ

<div class="flow">
App.tsx         = khung 3 cột + header + modal
store.ts       = bộ nhớ trung tâm
api.ts         = cổng nối backend
SearchPanel    = nhập query/filter/mode
ResultGrid     = danh sách kết quả
ResultCard     = từng frame
RecheckPanel   = chọn đáp án + CSV
PreviewModal   = xem frame lớn
VideoModal     = xem video nếu backend có
index.css      = màu, scrollbar, global style
</div>

<div class="ok">
Nếu không biết sửa ở đâu, hãy bắt đầu bằng cách tìm tên chữ đang hiển thị trên UI trong toàn project. Ví dụ tìm “News Retrieval v2”, “ANSWER BUILDER”, “SEARCH”, “COLS”, “No results”. VS Code sẽ đưa bạn tới đúng component.
</div>